# MiniCPM-o-2.6 Local Deployment Guide

This notebook provides a complete guide to download, fix, and run MiniCPM-o-2.6 model locally on Google Colab or similar environments.

## Requirements
- GPU: Tesla L4 or better (22GB+ VRAM)
- Python 3.10+
- CUDA-compatible PyTorch

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install --upgrade transformers accelerate torch pillow numpy -q

print("Dependencies installed successfully")

## Step 2: Download Model from Hugging Face

This will download the complete MiniCPM-o-2.6 model (~16-20 GB) to your local directory.

In [ ]:
from huggingface_hub import snapshot_download
import os

# Set download directory
model_dir = "/content/MiniCPM-o-2_6"

print("📥 Downloading MiniCPM-o-2.6 model...")
print("⏳ This may take 10-30 minutes depending on your network speed\n")

model_path = snapshot_download(
    repo_id="openbmb/MiniCPM-o-2_6",
    local_dir=model_dir,
    local_dir_use_symlinks=False,
    resume_download=True
)

print(f"\n✅ Model downloaded to: {model_path}")

# List downloaded files
files = os.listdir(model_path)
print(f"\n📁 Total files: {len(files)}")
print(f"💾 Directory size: {sum(os.path.getsize(os.path.join(model_path, f)) for f in files if os.path.isfile(os.path.join(model_path, f))) / (1024**3):.2f} GB")

## Step 3: Fix Image Processing Bug

There's a known bug in the image normalization function that causes shape mismatch errors. We need to fix it before running the model.

In [ ]:
import json
import shutil
from pathlib import Path

print("🔧 Applying fixes to the model...\n")

# Fix 1: Disable TTS module (not needed for vision tasks)
config_path = f"{model_dir}/config.json"
with open(config_path, "r") as f:
    config = json.load(f)

config["init_tts"] = False

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("✅ Disabled TTS module")

# Fix 2: Patch image processing normalize function
img_proc_file = f"{model_dir}/image_processing_minicpmv.py"

with open(img_proc_file, "r") as f:
    content = f.read()

# Replace problematic normalize call
old_code = '''                image_patches = [
                    self.normalize(image=image, mean=self.mean, std=self.std, input_data_format=input_data_format)
                    for image in image_patches
                ]'''

new_code = '''                # Manual normalize to avoid transformers bug
                normalized_patches = []
                for img_patch in image_patches:
                    mean_np = np.array([0.5, 0.5, 0.5], dtype=img_patch.dtype)
                    std_np = np.array([0.5, 0.5, 0.5], dtype=img_patch.dtype)
                    normalized = (img_patch - mean_np) / std_np
                    normalized_patches.append(normalized)
                image_patches = normalized_patches'''

content = content.replace(old_code, new_code)

with open(img_proc_file, "w") as f:
    f.write(content)

print("✅ Patched image processing bug")

# Clean cache to ensure fixes are applied
cache_paths = [
    "/root/.cache/huggingface/modules/transformers_modules",
    "/content/cache/modules/transformers_modules"
]

for cache_path in cache_paths:
    if Path(cache_path).exists():
        shutil.rmtree(cache_path)

print("✅ Cleared model cache\n")
print("🎉 All fixes applied successfully!")

## Step 4: Load Model and Tokenizer

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import random
import numpy as np
import os
from transformers import set_seed as hf_set_seed

# Set random seed for reproducibility
def set_seed(seed=17):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    hf_set_seed(seed)

set_seed(17)

print("🚀 Loading model...")
print("⏳ This may take 1-2 minutes\n")

model = AutoModel.from_pretrained(
    model_dir,
    trust_remote_code=True,
    attn_implementation='sdpa',
    torch_dtype=torch.bfloat16
)
model = model.eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)

print("✅ Model loaded successfully!")
print(f"📊 GPU memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

## Step 5: Run Multi-turn Conversation Inference

This example demonstrates a two-round conversation:
1. First round: Ask the model to describe the image
2. Second round: Ask for a summary based on the first response

In [ ]:
from PIL import Image

# Load your image
image_path = "your-photo-path"  # Replace with your image path
image = Image.open(image_path).convert('RGB')

print(f"📷 Loaded image: {image_path}")
print(f"   Size: {image.size}")
print(f"   Mode: {image.mode}\n")

# ========== Round 1: Describe the image ==========
print("🔮 Round 1: Asking model to describe the image...\n")

msgs_round1 = [
    {"role": "system", "content": "You are a helpful multimodal assistant that can analyze and describe pictures in detail."},
    {"role": "user", "content": [image, "Please describe what you see in this image."]}
]

with torch.no_grad():
    answer_round1 = model.chat(
        image=image,
        msgs=msgs_round1,
        tokenizer=tokenizer,
        temperature=0,
        max_new_tokens=200,
        do_sample=False,
        use_cache=False
    )

print("="*60)
print("📝 Round 1 Response:")
print("="*60)
print(answer_round1)
print("="*60)

# ========== Round 2: Summarize based on first response ==========
print("\n🔮 Round 2: Asking for a summary...\n")

msgs_round2 = [
    {"role": "system", "content": "You are a helpful multimodal assistant that can analyze and describe pictures in detail."},
    {"role": "user", "content": [image, "Please describe what you see in this image."]},
    {"role": "assistant", "content": answer_round1},  # Use model's own response
    {"role": "user", "content": "Now summarize the main objects and their positions briefly."}
]

with torch.no_grad():
    answer_round2 = model.chat(
        image=image,
        msgs=msgs_round2,
        tokenizer=tokenizer,
        temperature=0,
        max_new_tokens=200,
        do_sample=False,
        use_cache=False
    )

print("="*60)
print("📝 Round 2 Response:")
print("="*60)
print(answer_round2)
print("="*60)

print("\n🎉 Inference completed successfully!")

## Additional Notes

### Key Issues Encountered and Fixed:

1. **Image Processing Bug**: The original `image_processing_minicpmv.py` has a broadcasting error in the normalize function. We replaced it with manual normalization.

2. **TTS Module**: Disabled by default as it's not needed for vision tasks and requires additional dependencies.

3. **Cache Issues**: Always clear the transformers module cache after modifying model files to ensure changes take effect.

### Performance Tips:

- Use `sdpa` attention for better performance (requires CUDA)
- Set `use_cache=False` for cleaner memory management in multi-turn conversations
- Adjust `max_new_tokens` based on your needs (longer responses = more tokens)

### Troubleshooting:

- **Out of Memory**: Try reducing batch size or image resolution
- **Slow inference**: Ensure you're using GPU and `torch.bfloat16` dtype
- **Different results each run**: Check if random seed is properly set